# Incubator Controller Log — Interactive Environmental & PID Analytics

**Separate from the Govee analysis notebook.** **Colab tip:** use the native Drive/Upload cells below when dashboard buttons do not respond. Load one or multiple controller CSV logs; cut by source CSV row number or device timestamp; inspect measured temperatures, RH, CO₂, STC31 and the signed `Ctrl` commands. All plots and tables are recalculated from the same selected time window.

**Data-integrity safeguards:** No measured readings are fabricated. RH=0 is **retained by default** and prominently flagged; use the *Mask RH=0* checkbox only if you have independently confirmed these readings are logger/sensor faults. Raw values remain in `RH_Original`. CO₂ and STC31 have **unknown units** and are shown as logged. Signed `Ctrl` values indicate high/positive versus low/negative output; their physical actuator direction is *not inferred*. Reference values shown below are **editable analysis examples, not verified firmware setpoints**.

> **Row convention:** Header is CSV row 1, first data reading is row 2, and both start/end rows are inclusive. When comparing multiple runs, prefer timestamp windows instead of equal row indices.


### 1. Install dependencies and load the analysis engine

In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "plotly>=5.18,<7" "ipywidgets>=8,<9"

In [ ]:
from pathlib import Path
import os, re, json, sys
from urllib.request import urlretrieve
import pandas as pd
import numpy as np
import ipywidgets as w
# Google Colab requires a separate bridge for Python widget callbacks.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    print("Colab widget callback bridge enabled")
except ImportError:
    print("Outside Colab; using local widget manager")
from IPython.display import display, HTML, clear_output

ENGINE = Path('incubator_controller_analytics.py')
repo_url = ('https://raw.githubusercontent.com/leotirto/'
            'Cell-Incubator-Chamber-Temperature-RH-Analytics/'
            'main/incubator_controller_analytics.py')
print('Refreshing current analysis engine from GitHub...')
urlretrieve(repo_url, str(ENGINE))
import importlib
if 'incubator_controller_analytics' in sys.modules:
    importlib.reload(sys.modules['incubator_controller_analytics'])
from incubator_controller_analytics import (
    read_controller_log, load_folder, build_report, plot_environment,
    plot_controller_signals, plot_pair, plot_statistics, plot_control_summary,
    plot_correlation, plot_lag_correlation, plot_excursions,
    lag_correlation, export_report, CONTROL_PAIRS, TIME, CTRL_COLS,
    detect_drop_recovery, plot_drop_recovery, plot_recovery_bars,
    compare_drop_recovery, plot_recovery_comparison, plot_recovery_summary_bars
)
print('Interactive analytics engine loaded.')

### 2. Interactive dashboard

Run setup and dashboard cells first. In Colab, **use the native Drive or Upload loader cells below** for reliable CSV ingestion. The Load button can reload already-mounted Drive folders; the native file chooser cannot safely run inside a widget callback. If buttons still fail, run the direct analysis code cell below.


In [ ]:
# DATA SOURCE
source_mode = w.Dropdown(options=[('Google Drive / local folder','folder'),('Upload CSV(s) in Colab','upload')], value='folder', description='Input:')
folder_path = w.Text(value='/content/drive/MyDrive/Envi Data', description='CSV folder:', layout=w.Layout(width='650px'))
load_button = w.Button(description='Load / reload CSVs', button_style='info', icon='folder-open', layout=w.Layout(width='190px'))
file_selector = w.Dropdown(options=[('Load files first','')], value='', description='Log file:', layout=w.Layout(width='900px'))
load_out = w.Output()

# WINDOW FILTERS
start_row = w.IntText(value=2, description='Start row:')
end_row = w.IntText(value=0, description='End row:', tooltip='0 = through end of file')
start_time = w.Text(value='', description='From time:', placeholder='YYYY-MM-DD HH:MM:SS')
end_time = w.Text(value='', description='To time:', placeholder='YYYY-MM-DD HH:MM:SS')
full_time_button = w.Button(description='Full time', icon='maximize', layout=w.Layout(width='125px'))
last_30_button = w.Button(description='Last 30 min', icon='clock-o', layout=w.Layout(width='140px'))
last_60_button = w.Button(description='Last 60 min', icon='clock-o', layout=w.Layout(width='140px'))
time_window_status = w.HTML(value='<small>Load a CSV, then select a time window.</small>')
mask_zero = w.Checkbox(value=False, description='Mask RH=0 (retain original)', indent=False)

# ANALYSIS REFERENCES — NOT CONFIRMED INCUBATOR SETPOINTS
target_t = w.FloatText(value=37.0, description='T target °C:')
tol_t = w.FloatText(value=0.5, description='T ± °C:')
target_rh = w.FloatText(value=90.0, description='RH target %:')
tol_rh = w.FloatText(value=1.0, description='RH ± %:')
co2_target = w.Text(value='', description='CO2 target:', placeholder='blank = disabled (raw units)')
co2_tolerance = w.Text(value='', description='CO2 ±:', placeholder='blank = disabled')
temp_threshold = w.FloatText(value=36.5, description='T ≥ °C:')
rh_threshold = w.FloatText(value=85.0, description='RH ≥ %:')
co2_threshold = w.FloatText(value=47000.0, description='CO2 ≥ raw:')

# METRIC / VISUALIZATION OPTIONS
hold_seconds = w.IntText(value=30, description='Hold (s):')
min_event_seconds = w.IntText(value=5, description='Event min (s):')
max_gap = w.FloatText(value=5.0, description='Max gap (s):')
neutral_band = w.FloatText(value=.02, description='Ctrl ±neutral:')
saturation_level = w.FloatText(value=.98, description='Ctrl saturation:')
smooth_seconds = w.IntSlider(value=60, min=0, max=300, step=10, description='Moving avg (s):', style={'description_width':'initial'}, layout=w.Layout(width='460px'))
resample = w.Dropdown(options=[('Raw readings', None),('5 s','5s'),('10 s','10s'),('30 s','30s'),('1 min','1min'),('5 min','5min')], value='10s', description='Plot bins:')
paired = w.Dropdown(options=list(CONTROL_PAIRS), value='RH', description='Pair:')
bar_group = w.Dropdown(options=['Temperature','RH','CO2','STC31'], value='Temperature', description='Bar group:')
lag_window = w.IntSlider(value=300, min=30, max=900, step=30, description='Lag max (s):', style={'description_width':'initial'}, layout=w.Layout(width='350px'))

analyze_button = w.Button(description='Analyze selected window', button_style='success', icon='bar-chart', layout=w.Layout(width='240px'))
batch_button = w.Button(description='Batch compare all loaded files', button_style='info', icon='table', layout=w.Layout(width='280px'))
export_button = w.Button(description='Export selected results ZIP', button_style='warning', icon='download', layout=w.Layout(width='260px'))

outputs = {key:w.Output(layout=w.Layout(min_height='120px')) for key in
           ['Overview','Profiles','PID controllers','Measured vs control','Bar comparison',
            'Excursions','Correlations & lag','Quality & processed','Batch comparison','Export','Drop recovery','Recovery comparison']}
tabs=w.Tab(children=list(outputs.values()))
for i,k in enumerate(outputs): tabs.set_title(i,k)
state={'logs':{},'report':None,'charts':{},'errors':{}}

sources = w.VBox([w.HTML('<h3>Data source</h3>'),
                  w.HBox([source_mode,folder_path]),
                  w.HBox([load_button,file_selector]),load_out])
filters = w.VBox([w.HTML('<h3>Window & data quality</h3>'),
                  w.HBox([start_row,end_row]),w.HBox([start_time,end_time]),
                  w.HBox([full_time_button,last_30_button,last_60_button]),
                  time_window_status,mask_zero])
refs = w.VBox([w.HTML('<h3>Editable analysis references (NOT verified controller setpoints)</h3>'),
               w.HBox([target_t,tol_t,target_rh,tol_rh]),
               w.HBox([co2_target,co2_tolerance]),
               w.HBox([temp_threshold,rh_threshold,co2_threshold,hold_seconds])])
plotopts = w.VBox([w.HTML('<h3>Control & plotting options</h3>'),
                   w.HBox([max_gap,neutral_band,saturation_level,min_event_seconds]),
                   w.HBox([smooth_seconds,resample]),
                   w.HBox([paired,bar_group,lag_window])])
accordion=w.Accordion(children=[sources,filters,refs,plotopts], selected_index=0)
for i,title in enumerate(['01 Load CSVs','02 Select rows / time','03 Reference bands','04 Charts / controls']):
    accordion.set_title(i,title)
display(w.VBox([w.HTML('<h2>Incubator environment & PID dashboard</h2>'),accordion,
                w.HBox([analyze_button,batch_button,export_button]),tabs]))


In [ ]:
def _preset_time_window(minutes=None):
    """Set inclusive timestamp fields from the selected log, without editing rows."""
    name = file_selector.value
    if name not in state['logs']:
        time_window_status.value = '<b style="color:#b45309">Load a controller CSV first.</b>'
        return
    df = state['logs'][name].frame
    if df.empty:
        time_window_status.value = '<b>No valid timestamp rows in this file.</b>'
        return
    end = df[TIME].max()
    begin = df[TIME].min() if minutes is None else max(df[TIME].min(),end-pd.Timedelta(minutes=minutes))
    start_time.value=begin.strftime('%Y-%m-%d %H:%M:%S')
    end_time.value=end.strftime('%Y-%m-%d %H:%M:%S')
    # Row limits are independent and should not further trim preset windows.
    start_row.value=2
    end_row.value=0
    time_window_status.value = (
        '<small>Selected <b>'+str(begin)+'</b> to <b>'+str(end)+
        '</b>. Click <b>Analyze selected window</b>.</small>')

full_time_button.on_click(lambda _: _preset_time_window())
last_30_button.on_click(lambda _: _preset_time_window(30))
last_60_button.on_click(lambda _: _preset_time_window(60))


# Event handlers. Execute once after the dashboard controls cell.
def _select_params():
    kwargs = dict(
        start_row=int(start_row.value) if start_row.value > 0 else None,
        end_row=int(end_row.value) if end_row.value > 0 else None,
        start_time=start_time.value.strip() or None,
        end_time=end_time.value.strip() or None,
        mask_rh_zeros=bool(mask_zero.value),
        smooth_seconds=int(smooth_seconds.value), max_gap_seconds=float(max_gap.value),
        neutral_band=float(neutral_band.value), saturation_level=float(saturation_level.value),
        chamber_target=float(target_t.value), chamber_tolerance=float(tol_t.value),
        rh_target=float(target_rh.value), rh_tolerance=float(tol_rh.value),
        hold_seconds=float(hold_seconds.value),
        excursion_min_seconds=float(min_event_seconds.value),
        temp_threshold=float(temp_threshold.value), rh_threshold=float(rh_threshold.value),
        co2_threshold=float(co2_threshold.value),
    )
    if bool(co2_target.value.strip()) != bool(co2_tolerance.value.strip()):
        raise ValueError('For CO2 events, enter both target AND tolerance (in raw logged units)')
    if co2_target.value.strip():
        kwargs['co2_target'] = float(co2_target.value)
        kwargs['co2_tolerance'] = float(co2_tolerance.value)
    if kwargs['rh_tolerance'] < 0 or kwargs['chamber_tolerance'] < 0:
        raise ValueError('Target tolerances must be nonnegative')
    return kwargs


def _load_files(_=None):
    with load_out:
        clear_output(wait=True)
        try:
            if source_mode.value == 'upload':
                # Colab native file upload dialogs need a direct code cell.
                folder = Path('/content/incubator_controller_input')
                print('Upload mode: run the native Upload CSV cell below the dashboard.')
                if not folder.is_dir() or not list(folder.glob('*.csv')):
                    raise FileNotFoundError('No uploaded CSVs yet. Run the Upload CSV cell below.')
            else:
                folder = Path(folder_path.value.strip()).expanduser()
                if not folder.is_dir():
                    raise FileNotFoundError(f'CSV folder not found: {folder}. Check the Drive mount and path.')
            print('Scanning:', folder.resolve())
            logs, failures = load_folder(folder)
            state['logs'],state['errors'] = logs,failures
            if logs:
                file_selector.options = [(name,name) for name in logs]
                print(f'Loaded {len(logs)} controller log(s). Choose one and click Analyze.')
                for name, log in logs.items():
                    print(f'  {name}: {len(log.frame):,} valid timestamp rows')
            else:
                file_selector.options=[('No compatible logs','')]
                print('No matching controller CSV files found.')
            for filename,error in failures.items():
                print(f'Ignored incompatible CSV: {filename} — {error}')
        except Exception as e:
            print(f'LOAD FAILED: {type(e).__name__}: {e}')


def _analyze(_=None):
    name=file_selector.value
    with outputs['Overview']:
        clear_output(wait=True)
        if name not in state['logs']:
            print('Load CSVs first, then select a file.')
            tabs.selected_index=0
            return
        try:
            report=build_report(state['logs'][name],**_select_params())
            state['report']=report
            df=report['data']
            print(f'{name} | {len(df):,} selected records | {df.Timestamp.min()} → {df.Timestamp.max()}')
            print('Analysis references are user-editable; they are NOT confirmed firmware setpoints.')
            print('CO2 and STC31 units: unknown (raw logger values).')
            print('RH zeros: '+ ('masked in metrics (original preserved)' if mask_zero.value else 'retained in statistics'))
            display(report['threshold'].T.rename(columns={0:'Result'}).round(3))
            display(report['stats'].round(3))
        except Exception as e:
            print(f'ANALYSIS FAILED: {type(e).__name__}: {e}')
            tabs.selected_index=0
            return

    charts={}
    work = [
      ('Profiles','profile',lambda:plot_environment(df,resample=resample.value,overlay_moving_average=True,targets=report['targets'])),
      ('PID controllers','pid',lambda:plot_controller_signals(df,resample=resample.value)),
      ('Measured vs control','pair',lambda:plot_pair(df,paired.value,resample=resample.value,
                                        target=report['targets'].get(paired.value))),
      ('Bar comparison','bars',lambda:plot_statistics(report['stats'],group=bar_group.value)),
    ]
    for tabname, key, fn in work:
        with outputs[tabname]:
            clear_output(wait=True)
            try:
                fig=fn()
                fig.show()
                charts[key]=fig
            except Exception as e:
                print(f'Chart error: {e}')
    with outputs['Bar comparison']:
        fig=plot_control_summary(report['controllers']); fig.show()
        charts['pid_sign_distribution']=fig
        display(report['controllers'].round(3))
    with outputs['Excursions']:
        clear_output(wait=True)
        events=report['events']
        print(f'{len(events)} out-of-band runs (min duration {min_event_seconds.value}s).')
        print('Recovery requires an observed return to band followed by the hold duration.')
        if not events.empty:
            display(events.sort_values('Start').head(75).round(3))
        else:
            print('No qualifying excursions in the selected window.')
        for label, (target,tol) in report['targets'].items():
            if label not in df:
                continue
            scoped=events[events.Channel == label] if not events.empty else events
            fig=plot_excursions(df,label,target,tol,scoped)
            fig.show()
            charts[f'excursions_{label.replace(" ","_")}']=fig
    with outputs['Correlations & lag']:
        clear_output(wait=True)
        print('Exploratory descriptive association only. Feedback control introduces confounding; lag is NOT proof of actuator response delay.')
        fig=plot_correlation(df);fig.show();charts['correlation']=fig
        meas=paired.value
        ctrl=CONTROL_PAIRS[meas]
        if ctrl in df and meas in df:
            try:
                lag_df=lag_correlation(df, measurement=meas, controller=ctrl,
                         bin_seconds=10, max_lag_seconds=int(lag_window.value), min_overlaps=15)
                fig=plot_lag_correlation(lag_df,measurement=meas,controller=ctrl)
                fig.show();charts['lag']=fig
                display(lag_df.dropna(subset=['Pearson_r']).sort_values('Pearson_r',key=lambda s:s.abs(),ascending=False).head(8).round(3))
            except Exception as e:
                print('Lag unavailable:',e)
    with outputs['Quality & processed']:
        clear_output(wait=True)
        print('Quality issues from the complete source file:')
        display(pd.DataFrame([report['quality']]).T.rename(columns={0:'Value'}))
        print('Selected processed records (first 25). Original row positions are preserved in SourceRow.')
        display(df.head(25).round(4))
        print('All selected records can be exported via the Export button.')
    with outputs['Drop recovery']:
        clear_output(wait=True)
        drops = report['drop_recovery']
        print('Recovery = first observed above-threshold sample with a subsequent sustained hold.')
        print(f'CO2 threshold: {co2_threshold.value:g} (raw logger units); hold: {hold_seconds.value}s.')
        print('An incomplete recovery is NaN, not zero. Left-censored drops begin before the visible window.')
        if drops.empty:
            print('No below-threshold drop episodes in selected rows/time.')
        else:
            display(drops.round(3))
            recovered = drops[drops.Recovered & drops.Channel.ne('Any below threshold')]
            if not recovered.empty:
                display(recovered.groupby('Channel')['Recovery_min'].agg(
                    ['count','mean','median','min','max']).round(3))
        thresholds={'T Chamber':float(temp_threshold.value),
                    'RH':float(rh_threshold.value),'CO2':float(co2_threshold.value)}
        fig=plot_drop_recovery(df,drops,thresholds)
        fig.show()
        charts['drop_timeline']=fig
        fig=plot_recovery_bars(drops)
        fig.show()
        charts['drop_recovery_bars']=fig
    with outputs['Recovery comparison']:
        clear_output(wait=True)
        summary = report['recovery_comparison']
        print('Time window: '+str(df[TIME].iloc[0])+' to '+str(df[TIME].iloc[-1]))
        print('Each channel uses its own below-threshold onset; RH and CO2 are not converted into temperature units.')
        print('Only confirmed, non-left-censored events contribute to the median comparison.')
        display(summary.round(3))
        good=summary.dropna(subset=['Median_recovery_min'])
        if len(good)>1:
            fastest=good.loc[good.Median_recovery_min.idxmin()]
            slowest=good.loc[good.Median_recovery_min.idxmax()]
            print(f'Fastest median: {fastest.Channel} ({fastest.Median_recovery_min:.2f} min)')
            print(f'Slowest median: {slowest.Channel} ({slowest.Median_recovery_min:.2f} min)')
            print(f'Median time difference: {slowest.Median_recovery_min-fastest.Median_recovery_min:.2f} min')
        thresholds={'T Chamber':float(temp_threshold.value),
                    'RH':float(rh_threshold.value),'CO2':float(co2_threshold.value)}
        fig=plot_recovery_comparison(df,report['drop_recovery'],thresholds)
        fig.show()
        charts['normalized_recovery_comparison']=fig
        fig=plot_recovery_summary_bars(summary)
        fig.show()
        charts['median_recovery_comparison']=fig
        print('Normalized deficit = (threshold - measurement) / largest deficit in the selected interval, clipped to 0–100%.')
        print('A value of zero means at or above threshold; normalized deficit is not a physical severity comparison.')
    state['charts']=charts
    tabs.selected_index=0


def _batch(_=None):
    with outputs['Batch comparison']:
        clear_output(wait=True)
        if not state['logs']:
            print('Load CSVs first.')
            tabs.selected_index=8
            return
        try:
            options=_select_params()
            rows=[]
            for name, log in state['logs'].items():
                try:
                    report=build_report(log,**options)
                    stat=report['stats'].set_index('Channel')
                    row={'File':name,'Selected samples':len(report['data']),
                         'Duration_min':(report['data'].Timestamp.iloc[-1]-report['data'].Timestamp.iloc[0]).total_seconds()/60,
                         'RH zeros (whole file)':log.quality.get('RH exact zeros'),
                         'RH excursions':sum(report['events'].Channel=='RH') if not report['events'].empty else 0}
                    for c in ['T Chamber','RH','CO2']:
                        if c in stat.index:
                            row[f'{c} mean']=stat.loc[c,'Mean']
                            row[f'{c} std']=stat.loc[c,'StdDev']
                            row[f'{c} time-in-band %']=stat.loc[c,'Time_in_band_pct']
                    for ch in ['T Chamber','RH','CO2']:
                        d=report['drop_recovery']
                        subset=d[d.Channel.eq(ch)]
                        row[f'{ch} drop episodes']=len(subset)
                        row[f'{ch} recovered']=int(subset.Recovered.sum())
                        row[f'{ch} median recovery min']=subset.Recovery_min.median()
                    rows.append(row)
                except Exception as e:
                    rows.append({'File':name,'Error':str(e)})
            state['batch']=pd.DataFrame(rows)
            display(state['batch'].round(3))
            print('Run-level comparisons are independent; timelines are never silently concatenated.')
        except Exception as e:
            print('Batch comparison failed:',e)
        tabs.selected_index=8


def _export(_=None):
    with outputs['Export']:
        clear_output(wait=True)
        report=state.get('report')
        if not report:
            print('Run Analyze first.')
            tabs.selected_index=9
            return
        try:
            from datetime import datetime
            from zipfile import ZipFile, ZIP_DEFLATED
            source_name = re.sub('[^A-Za-z0-9_-]','_',report['source'])[:100]
            outroot=Path('/content/controller_analytics_exports') if Path('/content').exists() else Path.cwd()/'controller_analytics_exports'
            folder=outroot/source_name
            export_report(report,folder,charts=state['charts'])
            if 'batch' in state:
                state['batch'].to_csv(folder/'batch_comparison.csv',index=False)
            archive=outroot/f'{source_name}_analysis.zip'
            with ZipFile(archive,'w',ZIP_DEFLATED) as zipf:
                for p in folder.rglob('*'):
                    if p.is_file(): zipf.write(p,arcname=str(p.relative_to(outroot)))
            print(f'Exported: {archive}')
            try:
                from google.colab import files
                files.download(str(archive))
            except ImportError:
                print('Notebook outside Colab: download the ZIP from the path above.')
        except Exception as e:
            print(f'Export failed: {e}')
        tabs.selected_index=9

load_button.on_click(_load_files)
analyze_button.on_click(_analyze)
batch_button.on_click(_batch)
export_button.on_click(_export)
print('Dashboard ready. For reliable loading, run a native Drive/Upload cell below; then Analyze.')


### 2A. Reliable CSV loading (native Colab code cells)

Run **one** of the two loader cells below after running the dashboard setup cells. These do **not** depend on clicking widget buttons. A Drive mount only makes files accessible; you still need to load them.

- **Drive:** edit `CSV_FOLDER`, then run the Drive loader cell. It prints the matching CSV count, compatible logs and errors.
- **Upload:** run the Colab upload cell directly; select the CSV files in the browser dialog.

After loading, choose the log above and click **Analyze selected window**, or run the fallback analysis cell below if widget callbacks remain unavailable.


In [ ]:
# LOAD CONTROLLER CSVs FROM GOOGLE DRIVE — run this code cell
CSV_FOLDER = '/content/drive/MyDrive/Envi Data'  # edit if needed
from pathlib import Path
from incubator_controller_analytics import load_folder

def register_controller_csv_folder(folder):
    folder = Path(folder).expanduser()
    if not folder.is_dir():
        raise FileNotFoundError(f'CSV folder not found: {folder}')
    paths = sorted(folder.glob('*.csv'))
    print(f'Scanning: {folder} | {len(paths)} CSV file(s) found')
    if not paths:
        print('No CSV files in this folder; check spelling and the exact directory.')
    logs, failures = load_folder(folder)
    state['logs'], state['errors'] = logs, failures
    if logs:
        file_selector.options = [(name,name) for name in logs]
        file_selector.value = next(iter(logs))
        print(f'Loaded {len(logs)} controller log(s):')
        for name, log in logs.items():
            print(f' - {name}: {len(log.frame):,} timestamp-valid rows')
    else:
        file_selector.options = [('No compatible controller logs','')]
        print('No compatible controller logs found in the selected folder')
    for name, error in failures.items():
        print(f'Ignored {name}: {error}')
    return logs, failures

if CSV_FOLDER.startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
source_mode.value = 'folder'
folder_path.value = CSV_FOLDER
logs, failures = register_controller_csv_folder(CSV_FOLDER)


In [ ]:
# UPLOAD CSV(s) — run this native Colab code cell (not a widget callback)
from pathlib import Path
from google.colab import files
UPLOAD_FOLDER = Path('/content/incubator_controller_input')
UPLOAD_FOLDER.mkdir(parents=True, exist_ok=True)
print('Select one or more controller CSV files in the browser dialog...')
uploaded = files.upload()
for name, payload in uploaded.items():
    filename = Path(name).name
    if filename.lower().endswith('.csv'):
        (UPLOAD_FOLDER/filename).write_bytes(payload)
    else:
        print('Ignored non-CSV:', filename)
print(f'Uploaded {len(uploaded)} item(s) | folder: {UPLOAD_FOLDER}')
source_mode.value = 'upload'
logs, failures = register_controller_csv_folder(UPLOAD_FOLDER)


### 2B. Direct analysis when Colab dashboard buttons do not work

Run the cell below to generate statistics, PID charts and temperature/RH line and bar charts without widget button callbacks. Edit the selected log, row range and targets as needed.


In [ ]:
# DIRECT ANALYSIS — normal code cell, no button clicks required
from IPython.display import display
from incubator_controller_analytics import (
    build_report, plot_environment, plot_controller_signals,
    plot_statistics, plot_control_summary, plot_drop_recovery, plot_recovery_bars,
    plot_recovery_comparison, plot_recovery_summary_bars
)
if not state['logs']:
    raise RuntimeError('No controller logs loaded. Run one of the loader cells above first.')
SELECTED_LOG = file_selector.value if file_selector.value in state['logs'] else next(iter(state['logs']))
START_ROW = 2   # header is CSV row 1
END_ROW = None  # set e.g. 3842 for inclusive crop
START_TIME = None # e.g. '2026-11-14 12:30:00'
END_TIME = None
report = build_report(
    state['logs'][SELECTED_LOG],
    start_row=START_ROW, end_row=END_ROW,
    start_time=START_TIME, end_time=END_TIME,
    chamber_target=37.0, chamber_tolerance=0.5,
    rh_target=90.0, rh_tolerance=1.0,
    temp_threshold=36.5, rh_threshold=85.0,
    mask_rh_zeros=False,
    co2_threshold=47000.0  # raw logger units
)
state['report'] = report
state['charts'] = {}
print(f'{SELECTED_LOG}: {len(report["data"]):,} selected readings')
display(report['stats'].round(3))
display(report['controllers'].round(3))
for key, fig in (
    ('profiles',plot_environment(report['data'],resample='10s',targets=report['targets'])),
    ('pid',plot_controller_signals(report['data'],resample='10s')),
    ('temp_bars',plot_statistics(report['stats'],group='Temperature')),
    ('rh_bars',plot_statistics(report['stats'],group='RH')),
    ('controller_summary',plot_control_summary(report['controllers'])),
    ('drop_profiles',plot_drop_recovery(report['data'],report['drop_recovery'],
                       {'T Chamber':36.5,'RH':85.0,'CO2':47000.0})),
    ('drop_recovery_bars',plot_recovery_bars(report['drop_recovery'])),
    ('normalized_compare',plot_recovery_comparison(report['data'],report['drop_recovery'])),
    ('recovery_summary_bars',plot_recovery_summary_bars(report['recovery_comparison']))
):
    state['charts'][key] = fig
    fig.show()

print('Below-threshold recovery events:')
display(report['drop_recovery'].round(3))

print('Independent recovery comparison for the selected time window:')
display(report['recovery_comparison'].round(3))


### 3. Interpretation guidance

- **+ / − controls:** positive and negative output levels, as reported by your firmware. Do not interpret a negative command as cooling or a positive command as heating without an actuator-direction map.
- **RH=0:** physically within a mathematical RH range, but sustained zeros in a high-RH chamber are suspicious; inspect raw sensor/pump logs to determine whether they are genuine disturbances, offline readings or an intentional test.
- **CO₂:** the controller log uses values near tens of thousands. The units are **not encoded in this CSV**, so they stay raw; do not automatically label 50,000 as 5% or 50,000 ppm.
- **STC31:** unit and sensor interpretation not supplied; this notebook plots it separately as raw data.
- **Temperature-zone range:** the chamber, base, door and port sensors measure different components. Their min–max difference is **not** equivalent to spatial chamber-air uniformity.
- **Correlation and lag:** feedback actions often respond to disturbances, so correlation is not evidence that a PID command caused the subsequent measured change.
- **Time-in-band:** calculated from observed intervals only, not interpolated across long timestamp gaps. Threshold stability/recovery holds require successive observed samples.
- **Data distribution:** source CSVs and generated reports are not part of the public GitHub repository; upload them to Colab/Drive privately.

- **Below-threshold recovery:** the `Drop recovery` tab identifies declines below the editable T Chamber / RH / CO2 thresholds (default CO2 = 47000 raw). Recovery is first sampled reading at/above threshold followed by the configured hold duration. Temporary rebounds are not counted. Unconfirmed and window-edge episodes are flagged; data gaps break the analysis.

- **Time-window comparison:** set start/end under `02 Select rows / time`, or click Full time / Last 30 min / Last 60 min. Then click Analyze and open the **Recovery comparison** tab. The new tab shows independent recovery summaries, median recovery bars, and normalized deficits on one time axis. Only recovered and non-left-censored events are used in median comparisons. A row/time window that starts during an existing drop cannot establish its true recovery onset.
